# ex-9 — two-factor sampler, data only

Minimal notebook: runs (or loads) the 2-factor growing-p sweep and hands back the raw
per-replicate DataFrame `df_p` — **no plotting, no derived columns, no sign folding** —
for hand verification. Same model / design / seed as ex-8, and the same cache file
(`ex8_df_p.parquet`), so the rows here are the identical replicates behind ex-8's figures.

**Column glossary** (one row per replicate × factor; `j` ∈ {1, 2} indexes the estimated
factor $h_j$):

| column | meaning |
|---|---|
| `n, p, j, rep` | design cell + replicate index |
| `sin2_j` | $\sin^2\angle(h_j, b_j)$ — total misalignment to the own population direction |
| `rhs`, `floor`, `rotation` | theory RHS, its floor term, and the $k{\times}k$ rotation $\sin^2\angle(\hat\nu_j, e_j)$ |
| `measured_out_of_subspace` | $\lVert\Pi^\perp h_j\rVert^2$ — mass outside the $(b_1,b_2)$ plane |
| `hb1`, `hb2` | $\langle b_1, h_j\rangle$, $\langle b_2, h_j\rangle$ — **raw signed** projection coordinates (eigh sign is arbitrary per replicate) |
| `nu1`, `nu2` | components of $\hat\nu_j$ in the $e$-basis (also raw signed) |

**Identities to verify by hand** (each holds per row, to machine precision):

1. $\mathrm{hb}_1^2 + \mathrm{hb}_2^2 = 1 - \texttt{measured\_out\_of\_subspace}$
2. $\sin^2_j = 1 - \mathrm{hb}_j^2$ (own coordinate: `hb1` for j=1, `hb2` for j=2)
3. $\mathrm{nu}_1^2 + \mathrm{nu}_2^2 = 1$
4. `rotation` $= 1 - \mathrm{nu}_j^2$

For plotting: $h_j$'s point in the $(b_1, b_2)$ plane is simply (`hb1`, `hb2`). To resolve
the arbitrary eigh sign, multiply each row's *pair* by one sign (e.g. the sign of its
largest-|·| coordinate) — never flip the two coordinates independently.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from loguru import logger
logger.remove()

import numpy as np
import pandas as pd

from fl_experiment_setup import ModelSpec, DesignSpec, BaseExperiment
from fl_experiment_runner import run_experiment
from sim_theorem_partii import Eq6RHSAnalysis
from factor_lab.analyses.spectral import compute_true_eigenvalues
from factor_lab.analyses import compute_sine_alignment

DATA_DIR = REPO_ROOT / "nb_outputs"

K = 2
model = ModelSpec(
    k_factors=K,
    factor_vols=[0.16, 0.08],
    beta_samplers=[
        {"distribution": "normal", "loc": 1.0, "scale": 0.5},
        {"distribution": "normal", "loc": 0.0, "scale": 1.0},
    ],
    idio_vol_sampler={"distribution": "constant", "value": 0.4},
)
HEAVY_TAIL = dict(
    factor_return_sampler={"distribution": "student_t", "df": 6, "loc": 0.0, "scale": 1.0},
    idio_return_sampler={"distribution": "student_t", "df": 5, "loc": 0.0, "scale": 1.0},
)
N_REPS, SEED = 1000, 20260511
print("ready")

ready


In [2]:
class Eq6RHSWithNu(Eq6RHSAnalysis):
    """Eq6RHSAnalysis plus the eigenvector coordinates of the k×k realized Gram."""

    def analyze(self, context) -> dict:
        result = super().analyze(context)
        k, n = context.k, context.T
        F = context.factor_returns.T
        if self.center:
            F = F - F.mean(axis=1, keepdims=True)
        c_half = np.sqrt((context.model.B ** 2).mean(axis=1))
        D_hat = (c_half[:, None] * (F @ F.T / n)) * c_half[None, :]
        vals, vecs = np.linalg.eigh(D_hat)
        result["nu_coords"] = vecs[:, np.argsort(vals)[::-1]]
        return result


class ProjectedSpectrumAnalysis:
    """Dual-Gram PCA: sin², out-of-subspace mass, and raw projection coordinates."""

    def __init__(self, population_loading_directions_b_pop, center: bool = True):
        self.population_loading_directions_b_pop = population_loading_directions_b_pop
        self.center = center

    def analyze(self, context) -> dict:
        k = context.k
        Y = context.security_returns.T
        if self.center:
            Y = Y - Y.mean(axis=1, keepdims=True)
        G = Y.T @ Y
        eigenvalues, eigenvectors = np.linalg.eigh(G)
        top_k = np.argsort(eigenvalues)[::-1][:k]
        sv = np.sqrt(np.maximum(eigenvalues[top_k], 0.0))
        H = (Y @ eigenvectors[:, top_k]) / np.where(sv > 1e-14, sv, 1.0)   # (p, k), unit cols
        sin2_angle, _ = compute_sine_alignment(
            self.population_loading_directions_b_pop, H.T)
        coeffs = self.population_loading_directions_b_pop @ H              # (k, k): ⟨b_i, h_j⟩
        in_subspace = (coeffs ** 2).sum(axis=0)
        return {"sin2_j": sin2_angle,
                "measured_out_of_subspace": 1.0 - in_subspace,
                "h_coords": coeffs}


class InSubRotationExperiment(BaseExperiment):
    def __init__(self, center: bool = True):
        self.center = center

    def cell_setup(self, model, n: int, p: int):
        _, b_pop = compute_true_eigenvalues(model, model.k)
        return [ProjectedSpectrumAnalysis(b_pop, center=self.center),
                Eq6RHSWithNu(center=self.center)]

    def record(self, n: int, p: int, merged: dict) -> list[dict]:
        k = len(merged["sin2_j"])
        C, W = merged["h_coords"], merged["nu_coords"]
        return [{
            "n": n, "p": p, "j": j + 1,
            "sin2_j": float(merged["sin2_j"][j]),
            "rhs": float(merged["rhs"][j]), "floor": float(merged["floor"][j]),
            "rotation": float(merged["rotation"][j]),
            "measured_out_of_subspace": float(merged["measured_out_of_subspace"][j]),
            **{f"hb{i + 1}": float(C[i, j]) for i in range(k)},
            **{f"nu{i + 1}": float(W[i, j]) for i in range(k)},
        } for j in range(k)]


design_p = DesignSpec(
    n_values=[63], p_values=[100, 500, 5000, 10000, 20000, 50000],
    n_reps=N_REPS, random_seed=SEED, sampling="nested", nest_time=True, **HEAVY_TAIL,
)

_path = DATA_DIR / "ex8_df_p.parquet"       # shared with ex-8: identical replicates
_REQ = {"hb1", "hb2", "nu1", "nu2", "rotation"}
df_p = None
if _path.exists():
    df_p = pd.read_parquet(_path)
    if not (_REQ <= set(df_p.columns)
            and sorted(df_p["p"].unique()) == sorted(design_p.p_values)
            and int(df_p.groupby(["n", "p", "j"]).size().max()) == N_REPS
            and df_p["j"].max() == K):
        df_p = None
        print("cache mismatch — re-running")
    else:
        print(f"loaded cached sweep {_path.name}")
if df_p is None:
    df_p = run_experiment(model, design_p, InSubRotationExperiment(), progress=True)
    df_p.to_parquet(_path)
    print(f"ran sweep → {_path.name}")

print(f"df_p: {len(df_p):,} rows | columns: {', '.join(df_p.columns)}")
df_p.head(8)

cache mismatch — re-running


replicate:   0%|          | 0/1000 [00:00<?, ?rep/s]

replicate:   0%|          | 1/1000 [00:00<02:00,  8.28rep/s]

replicate:   0%|          | 2/1000 [00:00<02:09,  7.74rep/s]

replicate:   0%|          | 3/1000 [00:00<02:08,  7.74rep/s]

...

replicate: 100%|█████████▉| 999/1000 [02:12<00:00,  7.62rep/s]

replicate: 100%|██████████| 1000/1000 [02:13<00:00,  7.57rep/s]

replicate: 100%|██████████| 1000/1000 [02:13<00:00,  7.52rep/s]

ran sweep → ex8_df_p.parquet
df_p: 12,000 rows | columns: n, p, j, sin2_j, rhs, floor, rotation, measured_out_of_subspace, hb1, hb2, nu1, nu2, rep


,n,p,j,sin2_j,rhs,floor,rotation,measured_out_of_subspace,hb1,hb2,nu1,nu2,rep
0,63,100,1,0.080439,0.067049,0.066971,0.000084,0.079056,-0.958937,-0.037187,-0.999958,-0.009154,0
1,63,100,2,0.452418,0.300749,0.300690,0.000084,0.448785,-0.060275,0.739988,0.009154,-0.999958,0
2,63,500,1,0.082110,0.072987,0.072900,0.000093,0.081802,-0.958066,-0.017571,-0.999953,-0.009647,0
3,63,500,2,0.336934,0.304402,0.304338,0.000093,0.336560,-0.019341,0.814289,0.009647,-0.999953,0
4,63,5000,1,0.073341,0.072374,0.072296,0.000084,0.073316,-0.962631,-0.004931,-0.999958,-0.009172,0
5,63,5000,2,0.316730,0.317720,0.317662,0.000084,0.316690,-0.006270,0.826602,0.009172,-0.999958,0
6,63,10000,1,0.072742,0.072466,0.072388,0.000084,0.072688,-0.962943,-0.007328,-0.999958,-0.009156,0
7,63,10000,2,0.318940,0.318543,0.318486,0.000084,0.318858,-0.009006,0.825264,0.009156,-0.999958,0
